# Bai Shopping Brain — first real Kaggle GPU train
Включи GPU и нажми **Run All**. Контур делает preflight → SFT → baseline/candidate eval → promotion gate → staged brain release. Qwen3 запускается в non-thinking режиме, чтобы не ломать JSON-only контракт. Базовая модель закреплена immutable revision в `student-v0.1.json`.

Если в Kaggle Inputs лежат reviewed `gold.jsonl` + отдельный `eval-gold.jsonl`, используются они. Иначе запускается безопасный deterministic bootstrap 500 train + 60 holdout; это настоящий GPU SFT-пилот, но не human-reviewed teacher Gold.


In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, 'Нужен GPU'
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/1024**3, 2), 'GiB')


In [ ]:
!pip -q install -U 'transformers>=4.51,<5' 'peft>=0.15,<1' 'datasets>=3,<5' accelerate bitsandbytes sentencepiece huggingface-hub
!rm -rf /kaggle/working/tamdeshevle
!git clone --depth 1 --branch main https://github.com/eneonstudio-dev/tamdeshevle.git /kaggle/working/tamdeshevle
%cd /kaggle/working/tamdeshevle
!git rev-parse HEAD


In [ ]:
from pathlib import Path
import subprocess, sys
inputs=Path('/kaggle/input')
evals=list(inputs.rglob('eval-gold.jsonl'))
golds=[p for p in inputs.rglob('gold.jsonl') if p not in evals]
if len(evals)==1 and golds:
    eval_gold=evals[0]; train_gold=sorted(golds); mode='reviewed-input'
else:
    seed=Path('/kaggle/working/bai_seed')
    subprocess.check_call(['node','teacher-lab/training/deterministic-seed.mjs',str(seed)])
    eval_gold=seed/'eval-gold.jsonl'; train_gold=[seed/'gold.jsonl']; mode='deterministic-bootstrap'
cmd=[sys.executable,'teacher-lab/training/kaggle_train_pipeline.py']
for p in train_gold: cmd += ['--gold',str(p)]
cmd += ['--eval-gold',str(eval_gold),'--out','/kaggle/working/bai_auto_train']
print('MODE:',mode)
if mode=='deterministic-bootstrap': print('NOTE: pilot trains on deterministic approved seed, not human-reviewed teacher outputs')
print('TRAIN GOLD:', *train_gold, sep='\n- '); print('EVAL GOLD:',eval_gold)
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
root=Path('/kaggle/working/bai_auto_train')
print('=== GPU PREFLIGHT ===')
print((root/'gpu-preflight.json').read_text() if (root/'gpu-preflight.json').is_file() else 'missing')
print('=== PIPELINE ===')
print((root/'pipeline-manifest.json').read_text())
if (root/'brain-release.json').is_file():
    print('=== STAGED BRAIN RELEASE ==='); print((root/'brain-release.json').read_text())
print('=== ARTIFACTS ===')
for p in sorted(root.glob('*.zip')): print('-',p,round(p.stat().st_size/1024**2,2),'MiB')
